# Phase IIIb — Starry Night reference cleaning and covariance-aware confirmation

This short notebook closes the two remaining Phase-III methodological checks without rerunning the expensive corpus extraction.

It does three things:

1. recomputes the scale-ablation metrics on the Phase-II leakage-clean validation subset;
2. checks whether the uploaded *The Starry Night* reproduction, or a near-duplicate of it, already appears inside the Van Gogh reference corpus;
3. repositions *The Starry Night* after that reference cleaning using both the original robust RMS distance and covariance-aware distances after PCA, with **Minimum Covariance Determinant (MCD)** as the robust primary method and **Ledoit-Wolf** as a regularized sensitivity analysis.

No claim here concerns authenticity, emotion, intention, or perception.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 1. Recover the previous results

Upload the three ZIP files produced by Phases I, II and III:

- `painting_geometry_first_results.zip`
- `painting_geometry_phase2_results.zip`
- `painting_geometry_phase3_results.zip`

Only the small files needed for Phase IIIb are extracted.


In [ ]:
from google.colab import files
import io, zipfile, pandas as pd, numpy as np

INPUT_DIR = Path("/content/phase3b_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

NEEDED = {
    "features_train_multiscale.csv",
    "features_test_multiscale.csv",
    "excluded_test_images.csv",
    "scale_ablation_predictions.csv",
    "scale_ablation_results.csv",
    "starry_night_position_summary.csv",
}

def extract_needed(blob: bytes, target: Path):
    found = set()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for member in z.namelist():
            base = Path(member).name
            if base in NEEDED:
                with z.open(member) as src, open(target / base, "wb") as dst:
                    dst.write(src.read())
                found.add(base)
    return found

missing_now = [x for x in NEEDED if not (INPUT_DIR / x).exists()]
if missing_now:
    print("Upload the Phase I, II and III ZIP files.")
    uploaded = files.upload()
    found = set()
    for name, blob in uploaded.items():
        if name.lower().endswith(".zip"):
            found |= extract_needed(blob, INPUT_DIR)
        elif Path(name).name in NEEDED:
            (INPUT_DIR / Path(name).name).write_bytes(blob)
            found.add(Path(name).name)
    print("Recovered:", sorted(found))

required = {
    "features_train_multiscale.csv",
    "features_test_multiscale.csv",
    "excluded_test_images.csv",
    "scale_ablation_predictions.csv",
    "scale_ablation_results.csv",
}
missing = [x for x in required if not (INPUT_DIR / x).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

TRAIN_CSV = INPUT_DIR / "features_train_multiscale.csv"
TEST_CSV = INPUT_DIR / "features_test_multiscale.csv"
EXCLUDED_TEST = INPUT_DIR / "excluded_test_images.csv"
SCALE_PRED = INPUT_DIR / "scale_ablation_predictions.csv"
SCALE_RESULTS = INPUT_DIR / "scale_ablation_results.csv"
OLD_STARRY_SUMMARY = INPUT_DIR / "starry_night_position_summary.csv"

print("Inputs ready.")


## 2. Leakage-clean scale ablation

The Phase-III SVMs are **not retrained**. We reuse their stored predictions and remove the Phase-II validation exclusions before recomputing Macro-F1, bootstrap intervals, and paired deltas.

The additional contrast `S1248 - S124` tests whether the coarsest scale adds measurable discrimination after scales 1, 2, and 4 are already present.


In [ ]:
PHASE3B_SCALE = REPO_DIR / "results" / "phase3b_scale_clean"
PHASE3B_SCALE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/recompute_clean_scale_metrics.py",
    "--predictions", str(SCALE_PRED),
    "--scale-results", str(SCALE_RESULTS),
    "--excluded-test", str(EXCLUDED_TEST),
    "--output-dir", str(PHASE3B_SCALE),
]
subprocess.run(cmd, check=True)

scale_clean = pd.read_csv(PHASE3B_SCALE / "scale_ablation_results_clean.csv")
scale_clean_delta = pd.read_csv(PHASE3B_SCALE / "scale_ablation_deltas_clean.csv")
display(scale_clean.sort_values("macro_f1", ascending=False))
display(scale_clean_delta)


## 3. Recover the raw image corpus and upload *The Starry Night*

The hash audit must compare the actual uploaded reproduction against the Van Gogh image files, not against feature vectors.


In [ ]:
import kagglehub

DATASET_DIR = Path(kagglehub.dataset_download("delayedkarma/impressionist-classifier-data"))
print("Dataset:", DATASET_DIR)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def artist_root_score(path: Path):
    artist_dirs = [p for p in path.iterdir() if p.is_dir()]
    n_artists, n_images = 0, 0
    for d in artist_dirs:
        imgs = [p for p in d.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        if imgs:
            n_artists += 1
            n_images += len(imgs)
    return n_artists, n_images

def find_split_root(name: str):
    candidates = [p for p in DATASET_DIR.rglob(name) if p.is_dir()]
    scored = [(artist_root_score(p), p) for p in candidates]
    scored = [x for x in scored if x[0][0] >= 2]
    if not scored:
        raise FileNotFoundError(f"Could not resolve artist root for split={name}")
    scored.sort(key=lambda x: (x[0][0], x[0][1]), reverse=True)
    return scored[0][1]

TRAIN_ROOT = find_split_root("training")
TEST_ROOT = find_split_root("validation")
print("Training root:", TRAIN_ROOT)
print("Validation root:", TEST_ROOT)


In [ ]:
print("Upload the SAME reliable reproduction of The Starry Night used in Phase III.")
uploaded = files.upload()

image_names = [name for name in uploaded if Path(name).suffix.lower() in IMAGE_EXTS]
if not image_names:
    raise FileNotFoundError("No supported image uploaded.")

chosen = image_names[0]
STARRY_PATH = Path("/content") / Path(chosen).name
STARRY_PATH.write_bytes(uploaded[chosen])
print("Using:", STARRY_PATH)


## 4. Starry-specific reference audit

The permissive screen flags a candidate when either pHash or dHash is close. Automatic removal is stricter:

`exact bytes OR (pHash <= 4 AND dHash <= 4)`.

The contact sheet is for visual inspection; the strict exclusions are what feed the covariance-aware analysis.


In [ ]:
PHASE3B_AUDIT = REPO_DIR / "results" / "phase3b_starry_audit"
PHASE3B_AUDIT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "scripts/audit_starry_reference.py",
    "--image", str(STARRY_PATH),
    "--train-root", str(TRAIN_ROOT),
    "--test-root", str(TEST_ROOT),
    "--artist", "VanGogh",
    "--screen-phash", "10",
    "--screen-dhash", "10",
    "--exclude-phash", "4",
    "--exclude-dhash", "4",
    "--output-dir", str(PHASE3B_AUDIT),
]
subprocess.run(cmd, check=True)

audit_summary = pd.read_csv(PHASE3B_AUDIT / "starry_reference_audit_summary.csv")
audit_candidates = pd.read_csv(PHASE3B_AUDIT / "starry_reference_candidates.csv")
audit_exclusions = pd.read_csv(PHASE3B_AUDIT / "starry_reference_exclusions.csv")
display(audit_summary)
display(audit_candidates.head(20))
print("Strict reference exclusions:", len(audit_exclusions))


In [ ]:
from IPython.display import display, Image as IPyImage

sheet = PHASE3B_AUDIT / "starry_reference_contact_sheet.jpg"
if sheet.exists():
    display(IPyImage(filename=str(sheet)))
else:
    print("No screened candidates; no contact sheet was created.")


## 5. Covariance-aware position within Van Gogh

The original descriptive distance treats each robustly standardized feature equally:

$$d_{RMS}(x)=\sqrt{\frac{1}{p}\sum_j z_j^2}.$$

Because curvature summaries are correlated, Phase IIIb also fits PCA on the Van Gogh reference geometry and retains the smallest number of components explaining at least 90% of the variance, capped at 20. In that reduced space it computes:

- **MCD robust Mahalanobis distance** — primary covariance-aware result;
- **Ledoit-Wolf Mahalanobis distance** — regularized sensitivity analysis.

The reference first removes the Phase-II validation exclusions and then any strict Starry-specific duplicate/near-duplicate matches.


In [ ]:
PHASE3B_STARRY = REPO_DIR / "results" / "phase3b_starry_covaware"
PHASE3B_STARRY.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/position_starry_night_covaware.py",
    "--image", str(STARRY_PATH),
    "--train", str(TRAIN_CSV),
    "--test", str(TEST_CSV),
    "--phase2-excluded-test", str(EXCLUDED_TEST),
    "--starry-reference-exclusions", str(PHASE3B_AUDIT / "starry_reference_exclusions.csv"),
    "--artist", "VanGogh",
    "--long-side", "512",
    "--variance-target", "0.90",
    "--max-pca-components", "20",
    "--output-dir", str(PHASE3B_STARRY),
]
subprocess.run(cmd, check=True)

cov_summary = pd.read_csv(PHASE3B_STARRY / "starry_covaware_position_summary.csv")
cov_methods = pd.read_csv(PHASE3B_STARRY / "starry_covariance_distance_methods.csv")
cov_features = pd.read_csv(PHASE3B_STARRY / "starry_feature_percentiles_after_reference_cleaning.csv")

display(cov_summary)
display(cov_methods)
display(cov_features.head(15))


## 6. Compare Phase III and Phase IIIb conclusions

What matters is not that all distance definitions return exactly the same percentile. The key robustness question is whether they agree qualitatively on whether *The Starry Night* is central/moderate or an extreme outlier within the chosen Van Gogh corpus.


In [ ]:
comparison = {
    "phase3_old_rms_percentile": np.nan,
    "phase3b_clean_rms_percentile": float(cov_summary.loc[0, "rms_robust_z_percentile"]),
    "phase3b_mcd_percentile": float(cov_summary.loc[0, "mcd_distance_percentile"]),
    "phase3b_ledoitwolf_percentile": float(cov_summary.loc[0, "ledoitwolf_distance_percentile"]),
    "starry_reference_exclusions": int(cov_summary.loc[0, "starry_reference_exclusions"]),
    "reference_n_after_all_exclusions": int(cov_summary.loc[0, "reference_n_after_all_exclusions"]),
}

if OLD_STARRY_SUMMARY.exists():
    old = pd.read_csv(OLD_STARRY_SUMMARY)
    if "distance_percentile_within_artist" in old.columns:
        comparison["phase3_old_rms_percentile"] = float(old.loc[0, "distance_percentile_within_artist"])

comparison_df = pd.DataFrame([comparison])
display(comparison_df)
comparison_df.to_csv(REPO_DIR / "results" / "phase3b_starry_percentile_comparison.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt

labels = ["RMS clean", "MCD", "Ledoit-Wolf"]
values = [
    comparison["phase3b_clean_rms_percentile"],
    comparison["phase3b_mcd_percentile"],
    comparison["phase3b_ledoitwolf_percentile"],
]

plt.figure(figsize=(7, 4.5))
plt.bar(labels, values)
plt.axhline(0.95, linestyle="--", linewidth=1)
plt.ylabel("Distance percentile within Van Gogh")
plt.ylim(0, 1)
plt.title("The Starry Night — sensitivity to multivariate distance definition")
plt.tight_layout()
fig_path = PHASE3B_STARRY / "Figure_starry_distance_sensitivity.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## 7. Package Phase IIIb outputs

Upload the resulting ZIP back to the chat for final verification. If the duplicate-clean RMS, MCD, and Ledoit-Wolf analyses all point to a non-extreme position, the main experimental chain can be frozen for manuscript writing.


In [ ]:
import json, shutil

metadata = {
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "phase3b_goal": "clean scale metrics + Starry-specific reference audit + covariance-aware positioning",
    "starry_filename": STARRY_PATH.name,
    "strict_hash_exclusion_rule": "exact OR (pHash<=4 AND dHash<=4)",
    "pca_variance_target": 0.90,
    "pca_max_components": 20,
}
meta_path = REPO_DIR / "results" / "phase3b_run_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

PACKAGE_DIR = Path("/content/painting_geometry_phase3b_results")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir()

for folder_name in ["phase3b_scale_clean", "phase3b_starry_audit", "phase3b_starry_covaware"]:
    src = REPO_DIR / "results" / folder_name
    if src.exists():
        shutil.copytree(src, PACKAGE_DIR / folder_name)

for extra in [
    REPO_DIR / "results" / "phase3b_starry_percentile_comparison.csv",
    meta_path,
]:
    if extra.exists():
        shutil.copy2(extra, PACKAGE_DIR / extra.name)

zip_path = shutil.make_archive(
    "/content/painting_geometry_phase3b_results",
    "zip",
    root_dir=PACKAGE_DIR,
)
print("Created:", zip_path)
files.download(zip_path)
